# ch13 练习 —— 连接与合并

> 数据：`device_defects.csv`（270 x 13，18 个台区）、
> `device_info.csv`（16 x 5，**少 2 个台区**）、`station_summary.csv`（18 x 3）、
> `load_curve.csv`（单台区 2160 小时）

**做题规则**：从上往下依次填，每个 `____` 处跑通再往下。
验收块 `assert` 不要改，它是你自己对账用的。

| 题号 | 考点 |
|---|---|
| TODO(1) | 四种 `how` 的行数与语义 |
| TODO(2) | 定位"主表有、维表没有"的键 |
| TODO(3) | `indicator=True` 的 `_merge` 列 |
| TODO(4) | 脏键导致静默 0 匹配 |
| TODO(5) | `validate` 拦下基数错误 |
| TODO(6) | 多对多的笛卡尔积膨胀 |
| TODO(7) | `suffixes` 只作用于非键同名列 |
| TODO(8) | 主表 + 两张维表的链式拼装 |
| TODO(9) | `concat` 的索引重复 |
| TODO(10) | `concat` 的列对齐（`join`） |
| TODO(11) | `merge_asof` 的 `backward` |
| TODO(12) | `merge_asof` 的 `forward` 与排序约束 |
| 综合① | 交付宽表 + 六项自检 |
| 综合② | 连接质量审计表 |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

DATA = Path("data")

# 主表：缺陷记录（270 行，18 个台区，每台区多行）
dev = pd.read_csv(DATA / "device_defects.csv")
# 维表：设备台账（16 行，故意少 2 个台区）
info = pd.read_csv(DATA / "device_info.csv")
# 汇总表：台区日均负荷（18 行，与主表台区完全对齐）
summary = pd.read_csv(DATA / "station_summary.csv")

KEY = "台区编号"

# 维表换个中文列名，练「不同列名」的连接
info_cn = info.rename(columns={"容量kVA": "容量"})

# 容量限值调整表：每半月调一次，给 merge_asof 用
limit = pd.DataFrame(
    {
        "生效时间": pd.to_datetime(
            ["2026-01-01", "2026-01-16", "2026-02-01", "2026-02-16", "2026-03-01", "2026-03-16"]
        ),
        "限值": [1000, 1050, 1100, 1120, 1150, 1180],
    }
).sort_values("生效时间").reset_index(drop=True)

# 负荷曲线：取单台区、按时间升序（merge_asof 要求左表键有序）
curve = pd.read_csv(DATA / "load_curve.csv", parse_dates=["时间戳"])
hourly = (
    curve[curve[KEY] == "STATION_A_01"]
    .sort_values("时间戳")
    .reset_index(drop=True)
)

print("pandas", pd.__version__)
print("dev", dev.shape, "| info", info.shape, "| summary", summary.shape, "| hourly", hourly.shape)

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def merge_report(result, base_rows):
    """算一条校验公式：结果行数相对左表的膨胀倍数。"""
    return round(len(result) / base_rows, 4)


def match_rate(result, col):
    """右表列的非空率——判断连接命中情况最快的方式。"""
    return round(float(result[col].notna().mean()), 4)


def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。

    返回 ("ok", 结果) 或 ("err", 异常类型名)。
    """
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


def audit_merge(left, right, key, probe_col):
    """对一次 many_to_one 的 left 连接做质量审计。

    返回形状 / 行数守恒 / 匹配率 / 是否低于 0.95 阈值。
    """
    result = left.merge(right, on=key, how="left", validate="many_to_one")
    rate = round(float(result[probe_col].notna().mean()), 4)
    return {
        "形状": result.shape,
        "行数守恒": len(result) == len(left),
        "匹配率": rate,
        "低命中": rate < 0.95,
    }


print("dev 台区数:", dev[KEY].nunique(), "| info 台区数:", info[KEY].nunique())

---

## 基础题

In [ ]:
# TODO(1)：把 dev 与 info 按 台区编号 做四种 how 的连接，行数存入 sizes 字典
#           （键为 "inner" / "left" / "right" / "outer"，值为结果行数）
# 提示：dev.merge(info, on=KEY, how=how) ；再用 len(...)
sizes = ____
# ---- 验收 ----
assert set(sizes) == {"inner", "left", "right", "outer"}
assert sizes["inner"] == 247, f"inner 应为 247 行，实际 {sizes['inner']}"
assert sizes["left"] == 270, f"left 应为 270 行（行数守恒），实际 {sizes['left']}"
assert sizes["right"] == 247, f"right 应为 247 行，实际 {sizes['right']}"
assert sizes["outer"] == 270, f"outer 应为 270 行（右表无独有键），实际 {sizes['outer']}"
assert sizes["left"] - sizes["inner"] == 23, "inner 比 left 少的 23 行就是被丢掉的记录"
print("TODO(1) 通过 ->", sizes, "| inner 丢掉", sizes["left"] - sizes["inner"], "行")

In [ ]:
# TODO(2)：找出「主表有、维表没有」的台区编号列表，存入 missing_stations（升序）
#           并统计这些台区在主表里共有多少行，存入 n_missing_rows
# 提示：sorted(set(dev[KEY]) - set(info[KEY])) ；dev[KEY].isin(missing_stations).sum()
missing_stations = ____
n_missing_rows = ____
# ---- 验收 ----
assert missing_stations == ["STATION_B_03", "STATION_F_02"], f"缺的台区不对：{missing_stations}"
assert n_missing_rows == 23, f"这两个台区共应有 23 行，实际 {n_missing_rows}"
assert dev[KEY].nunique() == 18, "主表应有 18 个台区"
assert info[KEY].nunique() == 16, "维表应有 16 个台区"
assert len(dev.merge(info, on=KEY, how="left")) - n_missing_rows == 247, "对账：left - 缺失行 = inner"
print("TODO(2) 通过 -> 缺", missing_stations, "共", n_missing_rows, "行")

In [ ]:
# TODO(3)：用 indicator=True 做 left 连接存入 flagged，
#           然后统计 _merge 列各取值的行数存入 merge_counts（dict）
# 提示：dev.merge(info, on=KEY, how="left", indicator=True) ；
#        flagged["_merge"].value_counts().to_dict()
flagged = ____
merge_counts = ____
# ---- 验收 ----
assert merge_counts.get("both") == 247, f"both 应为 247，实际 {merge_counts.get('both')}"
assert merge_counts.get("left_only") == 23, f"left_only 应为 23，实际 {merge_counts.get('left_only')}"
assert merge_counts.get("right_only") == 0, \
    "右表无独有键，right_only 应为 0（pandas 3.0 的 _merge 是 category，0 计数也会列出）"
assert merge_counts["both"] + merge_counts["left_only"] == len(dev), "两部分之和应等于左表行数"
assert str(flagged["_merge"].dtype) == "category", "_merge 在 pandas 3.0 是 category 类型"
# indicator 列名可以自定义
named = dev.merge(info, on=KEY, how="left", indicator="来源")
assert "来源" in named.columns, "indicator 可传自定义列名"
print("TODO(3) 通过 ->", merge_counts)
print("   注意：_merge 是 category，value_counts 会把 0 计数的类别也列出来")

In [ ]:
# TODO(4)：演示脏键的静默失败：把 info 的台区编号前置一个空格存入 info_dirty，
#           分别做 left 连接，把「所属供电所」的匹配行数存入 n_clean / n_dirty，
#           并把脏键 strip 后复测的匹配行数存入 n_fixed
# 提示：info_dirty = info.copy() 后改列；用 result["所属供电所"].notna().sum()
info_dirty = ____
info_dirty[KEY] = ____
n_clean = ____
n_dirty = ____
n_fixed = ____
# ---- 验收 ----
assert n_clean == 247, f"干净键应匹配 247 行，实际 {n_clean}"
assert n_dirty == 0, f"脏键应匹配 0 行（静默失败），实际 {n_dirty}"
assert n_fixed == 247, f"strip 后应恢复 247 行，实际 {n_fixed}"
# 关键：形状完全一样，形状检查发现不了这个错误
assert dev.merge(info_dirty, on=KEY, how="left").shape == dev.merge(info, on=KEY, how="left").shape, \
    "脏键与干净键的结果形状应完全相同（所以才叫静默失败）"
print("TODO(4) 通过 -> 干净", n_clean, "| 脏键", n_dirty, "| strip 后", n_fixed,
      "| 形状相同:", True)

In [ ]:
# TODO(5)：用 validate 拦下基数错误：
#           对 dev 与 info 做 left 连接，验证 many_to_one 是否通过（布尔量存入 ok_many_to_one）；
#           再用 probe 探测 one_to_one（脚手架里的 probe 会返回 ("err", 异常类名)），存入 probe_one_to_one
# 提示：dev.merge(info, on=KEY, how="left", validate="many_to_one")
#        probe(dev.merge, info, on=KEY, how="left", validate="one_to_one")
ok_many_to_one = ____
probe_one_to_one = ____
# ---- 验收 ----
assert ok_many_to_one is True, "many_to_one 校验应通过"
assert probe_one_to_one[0] == "err", "one_to_one 应被拦下（左表键重复）"
assert probe_one_to_one[1] == "MergeError", f"应抛 MergeError，实际 {probe_one_to_one[1]}"
assert int(dev[KEY].duplicated().sum()) == 252, "左表 252 行有重复键，所以不是一对一"
print("TODO(5) 通过 -> many_to_one 通过 | one_to_one 被拦下:", probe_one_to_one[1])

In [ ]:
# TODO(6)：造两把小表演示多对多的笛卡尔积：a = {k:['x','x','y'], v:[1,2,3]}、
#           b = {k:['x','x','y'], w:[9,8,7]}，连接结果存入 blow，
#           形状存入 blow_shape，k=='x' 的行数存入 n_x
# 提示：a.merge(b, on="k")
a = ____
b = ____
blow = ____
blow_shape = ____
n_x = ____
# ---- 验收 ----
assert blow_shape == (5, 3), f"笛卡尔积结果应为 (5, 3)，实际 {blow_shape}"
assert n_x == 4, f"k=='x' 应为 2x2=4 行，实际 {n_x}"
assert len(a) + len(b) != len(blow), "多对多连接不能按「相加」估行数"
print("TODO(6) 通过 -> a(3) × b(3) →", blow_shape, "| k=='x' 有", n_x, "行（2x2 笛卡尔积）")

In [ ]:
# TODO(7)：造两把同名列的小表做带 suffixes 的连接：
#           取 dev 前三行的 记录ID / 台区编号 / 负荷值 两份（left3 / right3），
#           用 suffixes=("_左","_右") 连接存入 sfx，列名列表存入 sfx_cols
# 提示：dev[["记录ID", KEY, "负荷值"]].head(3) ；再 .merge(right3, on=KEY, suffixes=(...))
left3 = ____
right3 = ____
sfx = ____
sfx_cols = ____
# ---- 验收 ----
assert sfx_cols == ["记录ID_左", KEY, "负荷值_左", "记录ID_右", "负荷值_右"], \
    f"suffixes 列名不对：{sfx_cols}"
assert "台区编号_左" not in sfx_cols and "台区编号_右" not in sfx_cols, \
    "连接键不应被加后缀"
assert sfx.shape == (3, 5), f"形状应为 (3, 5)，实际 {sfx.shape}"
print("TODO(7) 通过 ->", sfx_cols)
print("   注意：键 '台区编号' 保持原名，其余同名列都加了后缀")

---

## 进阶题

In [ ]:
# TODO(8)：主表 + 两张维表链式拼装：dev ← info_cn ← summary（都 left、都 validate="many_to_one"），
#           结果存入 wide；形状存入 wide_shape；新增列列表（按出现顺序）存入 new_cols
# 提示：dev.merge(info_cn, on=KEY, how="left", validate="many_to_one").merge(summary, ...)
#        新增列 = [c for c in wide.columns if c not in dev.columns]
wide = ____
wide_shape = ____
new_cols = ____
# ---- 验收 ----
assert wide_shape == (270, 19), f"宽表应为 (270, 19)，实际 {wide_shape}"
assert new_cols == ["所属供电所", "容量", "投运年份", "供电区域类型", "日均负荷", "户数"], \
    f"新增列不对：{new_cols}"
assert wide_shape[1] == 13 + 5 + 3 - 1 - 1, "列数应等于 13 + 5 + 3 - 1 - 1"
assert len(wide) == len(dev), "行数必须守恒"
assert int(wide[KEY].nunique()) == 18, "台区数不能膨胀"
assert int(wide["容量"].isna().sum()) == 23, "台账列应缺 23 行"
assert int(wide["户数"].isna().sum()) == 0, "汇总表有全部 18 个台区，不该缺"
print("TODO(8) 通过 ->", wide_shape, "| 新增", len(new_cols), "列 | 台账缺 23 / 汇总缺 0")

In [ ]:
# TODO(9)：把 dev 前三行与后两行纵向 concat：
#           默认结果索引存入 idx_plain，ignore_index 结果索引存入 idx_fixed
#           （都转成 list 方便断言）
# 提示：pd.concat([c1, c2]) ；pd.concat([c1, c2], ignore_index=True)
c1 = ____
c2 = ____
idx_plain = ____
idx_fixed = ____
# ---- 验收 ----
assert idx_plain == [0, 1, 2, 0, 1], f"默认应保留原索引（有重复），实际 {idx_plain}"
assert idx_fixed == [0, 1, 2, 3, 4], f"ignore_index=True 应重排索引，实际 {idx_fixed}"
assert len(idx_plain) == 5 and len(idx_fixed) == 5, "两种写法行数应相同"
assert pd.concat([c1, c2]).index.has_duplicates, "默认结果索引必有重复"
assert not pd.concat([c1, c2], ignore_index=True).index.has_duplicates, "重排后不应有重复"
print("TODO(9) 通过 -> 默认索引", idx_plain, "| ignore_index 索引", idx_fixed)

In [ ]:
# TODO(10)：列不一致的 concat：p1 = dev 前三行的 [记录ID, 台区编号]，
#            p2 = dev 前三行的 [记录ID, 负荷值]；
#            默认结果形状存入 shape_outer、列名存入 cols_outer；
#            join="inner" 结果形状存入 shape_inner
# 提示：pd.concat([p1, p2]) ；pd.concat([p1, p2], join="inner")
p1 = ____
p2 = ____
outer_cat = ____
shape_outer = ____
cols_outer = ____
shape_inner = ____
# ---- 验收 ----
assert shape_outer == (6, 3), f"默认并集应为 (6, 3)，实际 {shape_outer}"
assert cols_outer == ["记录ID", KEY, "负荷值"], f"默认应取列并集，实际 {cols_outer}"
assert shape_inner == (6, 1), f"join='inner' 应只留共同列，实际 {shape_inner}"
assert int(outer_cat[KEY].isna().sum()) == 3, "p2 缺 台区编号，应补 3 个 NaN"
assert int(outer_cat["负荷值"].isna().sum()) == 3, "p1 缺 负荷值，应补 3 个 NaN"
print("TODO(10) 通过 -> 默认", shape_outer, cols_outer, "| inner", shape_inner)

In [ ]:
# TODO(11)：用 merge_asof(backward) 给 hourly 挂上"当时生效的限值"：
#            结果存入 asof，形状存入 asof_shape，限值 NaN 数存入 n_asof_nan，
#            限值取值分布（dict）存入 limit_dist，超限条数存入 n_over
# 提示：pd.merge_asof(hourly, limit, left_on="时间戳", right_on="生效时间", direction="backward")
#        分布用 asof["限值"].value_counts().sort_index().to_dict()
#        超限用 (asof["负荷值"] > asof["限值"]).sum()
asof = ____
asof_shape = ____
n_asof_nan = ____
limit_dist = ____
n_over = ____
# ---- 验收 ----
assert asof_shape == (2160, 7), f"应为 (2160, 7)，实际 {asof_shape}"
assert n_asof_nan == 0, f"backward 应能匹配到全部小时，实际 {n_asof_nan} 个 NaN"
assert limit_dist == {1000: 360, 1050: 384, 1100: 360, 1120: 312, 1150: 360, 1180: 384}, \
    f"限值区间分布不对：{limit_dist}"
assert sum(limit_dist.values()) == 2160, "各区间小时数之和应等于总小时数"
assert n_over == 1, f"超限应只有 1 条，实际 {n_over}"
print("TODO(11) 通过 ->", asof_shape, "| NaN", n_asof_nan, "| 超限", n_over, "条")
print("   限值分布:", limit_dist)

In [ ]:
# TODO(12)：对比 forward：用 merge_asof(direction="forward") 存入 fwd，
#            限值 NaN 数存入 n_fwd_nan；并用 probe 探测「未排序」的情况存入 probe_unsorted
# 提示：pd.merge_asof(hourly, limit, left_on="时间戳", right_on="生效时间", direction="forward")
#        probe(pd.merge_asof, hourly.sample(frac=1, random_state=0), limit,
#              left_on="时间戳", right_on="生效时间")
fwd = ____
n_fwd_nan = ____
probe_unsorted = ____
# ---- 验收 ----
assert n_fwd_nan == 383, f"forward 应有 383 个 NaN（最后一个生效时间之后），实际 {n_fwd_nan}"
assert fwd.shape == (2160, 7), "形状应与 backward 相同"
assert int(fwd["限值"].isna().sum()) > 0, "forward 会用到未来数据，末尾必然匹配不上"
assert probe_unsorted[0] == "err", "未排序时 merge_asof 应报错"
assert probe_unsorted[1] == "ValueError", f"应抛 ValueError，实际 {probe_unsorted[1]}"
assert n_fwd_nan > n_asof_nan, "forward 的 NaN 数应远多于 backward"
print("TODO(12) 通过 -> backward NaN", n_asof_nan, "| forward NaN", n_fwd_nan,
      "| 未排序 ->", probe_unsorted[1])

---

## 综合题

### 综合 ① 交付一张干净的宽表并自检

要求同时满足：行数守恒、键不膨胀、**无 `_x`/`_y` 残留**、`RangeIndex`、无重复索引。

In [ ]:
# TODO(13)：综合①：产出 deliver（dev + info_cn + summary 的宽表，已清列名、已 reset_index），
#          把六项自检结果存入 audit 字典：
#          行数 / 台区数 / 列数 / 台账缺行数 / 汇总缺行数 / 无后缀残留
# 提示：deliver.columns = [c.replace("_x","").replace("_y","") for c in deliver.columns]
#        deliver = deliver.reset_index(drop=True)
deliver = ____
deliver.columns = ____
deliver = ____
audit = ____
# ---- 验收 ----
assert audit["行数"] == 270, f"行数应守恒（270），实际 {audit['行数']}"
assert audit["台区数"] == 18, f"台区数应守恒（18），实际 {audit['台区数']}"
assert audit["列数"] == 19, f"列数应为 19，实际 {audit['列数']}"
assert audit["台账缺行数"] == 23, f"台账应缺 23 行，实际 {audit['台账缺行数']}"
assert audit["汇总缺行数"] == 0, f"汇总不该缺，实际 {audit['汇总缺行数']}"
assert audit["无后缀残留"] is True, "列名不应残留 _x / _y / _左 / _右"
assert not deliver.index.has_duplicates, "索引不应有重复"
assert isinstance(deliver.index, pd.RangeIndex), "索引应是 RangeIndex"
# 交付前的最后一次对账：原列一个不少
assert set(dev.columns).issubset(set(deliver.columns)), "原表列不得丢失"
print("综合① 通过 -> 交付宽表", deliver.shape)
for name, value in audit.items():
    print(f"   {name:<12} = {value}")

### 综合 ② 连接质量审计：把"命中率异常"自动报出来

写一段可复用的审计：给定主表、维表和键，自动给出
形状、行数守恒、匹配率、以及**匹配率是否低于阈值**的判断。
用这份数据实测：台账应为 **0.9148**（低于 0.95 阈值），汇总应为 **1.0**。

In [ ]:
# TODO(14)：综合②：用脚手架里的 audit_merge 对台账与汇总各跑一次审计，
#          结果存入 audit_table（键为 "台账" / "汇总"）
# 提示：audit_table = {"台账": audit_merge(dev, info_cn, KEY, "所属供电所"),
#                       "汇总": audit_merge(dev, summary, KEY, "日均负荷")}
audit_table = ____
# ---- 验收 ----
assert set(audit_table) == {"台账", "汇总"}
assert audit_table["台账"]["形状"] == (270, 17)
assert audit_table["汇总"]["形状"] == (270, 15)
assert audit_table["台账"]["行数守恒"] is True and audit_table["汇总"]["行数守恒"] is True
assert audit_table["台账"]["匹配率"] == 0.9148, f"台账匹配率应为 0.9148，实际 {audit_table['台账']['匹配率']}"
assert audit_table["汇总"]["匹配率"] == 1.0, f"汇总匹配率应为 1.0，实际 {audit_table['汇总']['匹配率']}"
assert audit_table["台账"]["低命中"] is True, "台账命中率 0.9148 < 0.95，应被标为低命中"
assert audit_table["汇总"]["低命中"] is False, "汇总命中率 1.0，不应被标为低命中"
# 关键结论：低命中不等于有错 —— 这里 0.9148 是因为 2 个台区业务上就没有台账
assert round(247 / 270, 4) == 0.9148, "0.9148 正好等于 247/270，说明缺的是那 23 行"
print("综合② 通过 -> 连接质量审计:")
for label, info_row in audit_table.items():
    print(f"   {label}: {info_row['形状']} | 行数守恒={info_row['行数守恒']} "
          f"| 匹配率={info_row['匹配率']} | 低命中={info_row['低命中']}")
print("   结论：台账低命中是「业务上确实没有」，不是键脏 —— 靠 indicator 区分")

---

### 复盘提问

**复盘问自己**（六题都能答上来才算过）：

1. 主表 + 维表为什么一律用 `how="left"`？用 `inner` 会丢什么？
2. 脏键（多一个空格）导致的失败有什么特征？为什么形状检查发现不了？
3. `validate` 参数不改变结果，那要它干什么？
4. `suffixes` 为什么不给连接键加后缀？列序有什么规律？
5. `pd.concat(axis=1)` 是按位置还是按标签对齐？`reset_index` 为什么必须？
6. `merge_asof` 的 `backward` 和 `forward` 分别代表"用过去"还是"用未来"？

答不上来的，回 `_notes/错题本.md` 记一笔。